In [53]:
import pandas as pd
import numpy as np
from imblearn.under_sampling import RandomUnderSampler


DATA_PATH = "/content/gdrive/MyDrive/REU/PROJECT/DATA/"

from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [54]:
annotated_data = pd.read_csv(DATA_PATH+"annotated_data.csv")

In [55]:
annotated_data

,Unnamed: 0,full_name,transactor_type
0,0,franklin county reagan coalition,Committee
1,1,nextgen climate action committee,Committee
2,2,zarwin baum good government pac,Committee
3,3,mid-atlantic laborers' political league,Committee
4,4,toll bros inc pac,Committee
...,...,...,...
518282,1048568,cindy olofson,Individual
518283,1048569,ryan olsen,Individual
518284,1048570,andrew orgill,Individual
518285,1048572,chris ortega,Individual


<h1>CONVERT TO BINARY LABELS</h1>

In [56]:
def org_labels(st):
    if st != "Individual" and st != "Candidate":
        return "Organization"
    return "Individual"

In [57]:
annotated_data['transactor_type'] = annotated_data['transactor_type'].apply(org_labels)
annotated_data = annotated_data[['full_name', 'transactor_type']].copy()

In [58]:
annotated_data.dropna(subset='full_name', inplace=True)

In [59]:
annotated_data['full_name'].sort_values()

,full_name
9580,citizens for boyle
8800,"mauck, shawn c"
360485,!jayne 2012
95841,"""""rip"""" stephen wilson"
323972,"""a company, inc. phx portable restooms"""
...,...
353475,zygmunt roguski
94328,zylphia cummins
193258,zymages
433906,zyra brown


<h1>Train, Test, split</h1>

In [60]:

def convert_bool(input_label):
    '''Converts to numerical encoding'''
    if input_label == 'Candidate' or input_label == 'Individual':
        return 1
    else:
        return 0

# encode transactor_type as numeric labels
annotated_data['label'] = np.vectorize(convert_bool)(annotated_data['transactor_type'])

# features and target for under-sampling
X = annotated_data[['full_name']]
y = annotated_data['label']

# apply random under-sampling to balance the classes
rus = RandomUnderSampler(random_state = 42)
X_resampled, y_resampled = rus.fit_resample(X, y)


In [61]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.25, random_state=24)


In [62]:
X_train

,full_name
33997,james lanigan
363131,banners on the cheap.com
289259,committee to elect barry gillaspie
206818,otis albert
6077,mt. lebanon democratic committee c/o geoffrey ...
...,...
443751,melinda t bishop-morfin
18613,sharon e huie-lew
7657,"newtown democrats, regina gairo, treasurer"
458214,driss ferza


In [63]:
y_train

,label
33997,1
363131,0
289259,0
206818,0
6077,0
...,...
443751,1
18613,1
7657,0
458214,1


In [64]:
X_test

,full_name
507706,quail creek crossing
225326,lyn t ward
141944,the governors
404422,briteverify
258567,u.s. american
...,...
228830,thomas russo
336128,colorado democratic party
300745,andreini-brophy anna
425221,jill blair


In [65]:
y_test

,label
507706,0
225326,1
141944,0
404422,0
258567,0
...,...
228830,1
336128,0
300745,1
425221,1


In [66]:
train_set = X_train.copy()
train_set['label'] = y_train
train_set

,full_name,label
33997,james lanigan,1
363131,banners on the cheap.com,0
289259,committee to elect barry gillaspie,0
206818,otis albert,0
6077,mt. lebanon democratic committee c/o geoffrey ...,0
...,...,...
443751,melinda t bishop-morfin,1
18613,sharon e huie-lew,1
7657,"newtown democrats, regina gairo, treasurer",0
458214,driss ferza,1


In [67]:
test_set = X_test.copy()
test_set['label'] = y_test
test_set

,full_name,label
507706,quail creek crossing,0
225326,lyn t ward,1
141944,the governors,0
404422,briteverify,0
258567,u.s. american,0
...,...,...
228830,thomas russo,1
336128,colorado democratic party,0
300745,andreini-brophy anna,1
425221,jill blair,1


In [68]:
train_set.to_csv("train.csv")
test_set.to_csv("test.csv")